# Agentes com LangGraph

Um agente decide sozinho quantos passos dar: chama o modelo, executa a ferramenta que o modelo pediu, devolve a observação e repete até não haver mais pedido. Este notebook escreve esse laço como um grafo, compara com o agente que o LangChain já traz montado, guarda o estado entre chamadas e termina montando um assistente de shell que conversa com o usuário.

In [ ]:
# No Google Colab, descomente e rode uma vez.

# !pip install -q langchain langchain-openai langgraph
# !pip install -q langchain-groq langchain-google-genai

# import os
# from google.colab import userdata

# os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
# os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
# os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
# os.environ["OPENROUTER_API_KEY"] = userdata.get("OPENROUTER_API_KEY")

In [ ]:
import subprocess
from pathlib import Path
from typing import Annotated, Literal

from IPython.display import Image
from typing_extensions import TypedDict

from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain.messages import HumanMessage, SystemMessage
from langchain.tools import tool
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import END, START, MessagesState, StateGraph
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition

In [ ]:
model = init_chat_model("openai:gpt-4.1-mini", temperature=0.0)


@tool
def calculate(expression: str) -> str:
    """Avalia uma expressão aritmética, como 12 * (3 + 4)."""
    return str(eval(expression, {"__builtins__": {}}, {}))

In [ ]:
tools = [calculate]
tools_by_name = {fn.name: fn for fn in tools}
model_with_tools = model.bind_tools(tools)

SYSTEM = "Você não faz contas de cabeça: toda conta é feita pela ferramenta."
REQUEST = "Quanto é 4871 vezes 3926, e quanto é a metade disso?"

## O laço como grafo

O laço do agente tem três responsabilidades: chamar o modelo, executar as chamadas que ele pediu e decidir se continua. As duas primeiras são nós, a terceira é uma aresta condicional que volta para o começo, e o estado que atravessa tudo é a lista de mensagens.

In [ ]:
class ChatState(TypedDict):
    messages: Annotated[list, add_messages]

In [ ]:
def call_model(state: ChatState) -> dict:
    """Chama o modelo com as ferramentas ligadas."""
    return {"messages": [model_with_tools.invoke([SystemMessage(SYSTEM)] + state["messages"])]}


def run_tools(state: ChatState) -> dict:
    """Executa as chamadas da última mensagem e devolve as observações."""
    calls = state["messages"][-1].tool_calls
    return {"messages": [tools_by_name[call["name"]].invoke(call) for call in calls]}


def should_continue(state: ChatState) -> Literal["run_tools", END]:
    """Vai para as ferramentas se o modelo pediu alguma, senão termina."""
    return "run_tools" if state["messages"][-1].tool_calls else END

In [ ]:
agent_builder = StateGraph(ChatState)

# nós
agent_builder.add_node("call_model", call_model)
agent_builder.add_node("run_tools", run_tools)

# arestas
agent_builder.add_edge(START, "call_model")
agent_builder.add_conditional_edges("call_model", should_continue, ["run_tools", END])
agent_builder.add_edge("run_tools", "call_model")

agent = agent_builder.compile()
Image(agent.get_graph().draw_mermaid_png())

In [ ]:
result = agent.invoke({"messages": [HumanMessage(REQUEST)]})

for message in result["messages"]:
    print(message.type, ":", message.tool_calls if getattr(message, "tool_calls", None) else message.content)

O ciclo aparece na conversa: a pergunta, a mensagem do modelo pedindo a ferramenta, uma observação por chamada e a resposta final. A aresta de `run_tools` de volta para `call_model` é o que separa um agente de um workflow, porque o número de voltas sai da decisão do modelo e não do código.

In [ ]:
asked = result["messages"][1]
observed = [message for message in result["messages"] if message.type == "tool"]

print(asked.tool_calls[0])
for message in observed:
    print(message.tool_call_id, "|", message.content)

Cada pedido tem `name`, `args` e um `id`. O `run_tools` passa a chamada inteira para `invoke`, e é isso que faz a `ToolMessage` voltar com o `tool_call_id` preenchido. O par é obrigatório: uma observação com um id que não existe no pedido anterior faz o provedor recusar a conversa com erro 400.

### Chamadas paralelas de ferramenta

Quando os pedidos não dependem um do outro, o modelo devolve várias chamadas na mesma mensagem. O `run_tools` já trata esse caso, porque percorre a lista de `tool_calls` e devolve uma observação por chamada.

In [ ]:
parallel = agent.invoke({"messages": [HumanMessage("Quanto é 12 vezes 12, e quanto é 15 mais 27?")]})

for message in parallel["messages"]:
    print(f'{message.type:6s} chamadas={len(getattr(message, "tool_calls", None) or [])} {str(message.content)[:45]!r}')

print("chamadas ao modelo:", sum(1 for message in parallel["messages"] if message.type == "ai"))

Uma mensagem do modelo trouxe as duas chamadas, e o grafo deu uma volta só no laço. Duas contas independentes custaram duas chamadas ao modelo, e não três, porque o modelo não precisou do primeiro resultado para pedir o segundo.

### A versão pronta

O LangGraph traz as duas peças genéricas do laço prontas: `ToolNode` executa as chamadas da última mensagem e devolve uma `ToolMessage` por chamada, e `tools_condition` decide entre o nó `tools` e o fim. O `MessagesState` é o estado com `add_messages` já anotado. Só o `call_model` continua escrito, porque é nele que estão o prompt e as ferramentas de cada agente.

In [ ]:
compact_builder = StateGraph(MessagesState)

# nós
compact_builder.add_node("call_model", call_model)
compact_builder.add_node("tools", ToolNode(tools))

# arestas
compact_builder.add_edge(START, "call_model")
compact_builder.add_conditional_edges("call_model", tools_condition)
compact_builder.add_edge("tools", "call_model")

compact_agent = compact_builder.compile()
print([message.type for message in compact_agent.invoke({"messages": [HumanMessage(REQUEST)]})["messages"]])

A conversa é a mesma do laço escrito à mão. O nó precisa se chamar `tools`, porque é esse nome que `tools_condition` devolve quando há chamadas pendentes. Entre escrever o laço inteiro e usar o agente pronto da próxima seção, este é o degrau do meio.

### Exercício 1

Acrescente uma segunda ferramenta ao `tools`, por exemplo uma que devolva o comprimento de um texto, e rode um pedido que precise das duas. Quantas voltas o laço deu, e o modelo pediu as duas ferramentas na mesma mensagem ou em mensagens separadas?

In [ ]:
# Seu código aqui

## O agente pronto

O `create_agent` monta esse mesmo grafo em uma linha, e monta também o prompt a cada passo. O `system_prompt` que ele recebe é inserido como primeira mensagem antes de cada chamada ao modelo, e cada observação volta como `ToolMessage` com o `tool_call_id` preenchido.

In [ ]:
prebuilt = create_agent(model, tools=tools, system_prompt=SYSTEM)

Image(prebuilt.get_graph().draw_mermaid_png())

In [ ]:
result = prebuilt.invoke({"messages": [HumanMessage(REQUEST)]})

print([message.type for message in result["messages"]])
print(result["messages"][-1].content)

O mesmo pedido do grafo escrito à mão dá o mesmo resultado. Não há mensagem de sistema no estado devolvido, porque o `system_prompt` é montado na hora da chamada e não fica guardado na conversa. Os nós do desenho se chamam `model` e `tools`, e fazem o que `call_model` e `run_tools` fazem acima.

## Persistência

Um checkpoint é o retrato do estado do grafo em um ponto da execução: as chaves, os valores que elas tinham e qual nó vem depois. O checkpointer é o objeto que grava esse retrato depois de cada nó, indexado pelo `thread_id` que vem na configuração da chamada.

In [ ]:
agent = agent_builder.compile(checkpointer=InMemorySaver())

config = {"configurable": {"thread_id": "sessao-1"}}

In [ ]:
result = agent.invoke({"messages": [HumanMessage("Eu moro em Natal. Quanto é 12 vezes 12?")]}, config)
print(result["messages"][-1].content)

result = agent.invoke({"messages": [HumanMessage("Onde eu moro?")]}, config)
print(result["messages"][-1].content)

A segunda chamada mandou só a pergunta nova. O grafo carregou o último checkpoint da thread e partiu do estado salvo, então o histórico já estava lá e o `add_messages` acrescentou a pergunta ao que existia.

In [ ]:
result = agent.invoke({"messages": [HumanMessage("Onde eu moro?")]}, {"configurable": {"thread_id": "sessao-2"}})
print(result["messages"][-1].content)

Outra `thread_id` parte do estado vazio e o agente não tem de onde tirar a resposta. Com o estado gravado em disco, o grafo também retoma de onde parou quando a execução falha no meio, o que um laço escrito à mão não consegue fazer.

### Onde os checkpoints ficam

O `InMemorySaver` guarda tudo em memória e some com o kernel, o que serve para experimentar. Em disco, o `SqliteSaver` grava em um arquivo único e basta para desenvolvimento, e o `PostgresSaver` aguenta acesso concorrente e é o de produção. Os dois vêm em pacotes próprios, nascem de um `from_conn_string` e pedem um `.setup()` na primeira vez, para criar as tabelas.

### Exercício 2

Continue a conversa da thread `sessao-1` com uma pergunta que só se responde com as duas mensagens anteriores. Quantas mensagens a thread acumulou depois dessa chamada?

In [ ]:
# Seu código aqui

## Assistente

Um assistente é um agente com memória de conversa e um laço de leitura em volta: a cada turno o usuário digita um pedido, o agente roda até responder e a resposta aparece na tela. O assistente abaixo tem uma ferramenta só, que roda comandos de shell na pasta deste notebook, e com ela responde perguntas sobre os arquivos que estão ali.

In [ ]:
@tool
def run_shell(command: str) -> str:
    """Roda um comando de shell na pasta atual e devolve a saída."""
    try:
        result = subprocess.run(command, shell=True, cwd=Path.cwd(), capture_output=True, text=True, timeout=20)
    except subprocess.TimeoutExpired:
        return "o comando passou de 20 segundos e foi interrompido"
    return (result.stdout + result.stderr).strip() or f"sem saída, código de retorno {result.returncode}"

In [ ]:
print(run_shell.invoke({"command": "ls | head -5"}))
print(run_shell.invoke({"command": "cat inexistente.txt"}))

A saída de erro volta junto com a saída normal, e é isso que deixa o agente ler a mensagem de um comando que falhou e tentar outro na volta seguinte do laço. O tempo limite existe porque o modelo às vezes roda algo que não termina.

### O laço com o usuário

O assistente é o agente pronto com a ferramenta e um checkpointer. A função `show` imprime o que cada passo produziu: o comando pedido, o começo da observação e a resposta final.

In [ ]:
ASSISTANT = (
    "Você é um assistente de shell. Responda perguntas sobre a pasta atual rodando comandos com run_shell, "
    "e responda ao usuário em uma ou duas frases a partir do que saiu."
)

assistant = create_agent(model, tools=[run_shell], system_prompt=ASSISTANT, checkpointer=InMemorySaver())

In [ ]:
def show(update: dict) -> None:
    """Imprime os comandos pedidos, as observações e a resposta de um passo."""
    for message in update["messages"]:
        if message.type == "tool":
            print(f"  saída: {message.content[:60]!r}")
        elif message.tool_calls:
            print("  comando:", [call["args"]["command"] for call in message.tool_calls])
        else:
            print("assistente:", message.content)

O `invoke` só devolve o estado final, depois que o laço inteiro rodou. O `stream` com `stream_mode="updates"` devolve um dicionário a cada nó que termina, com uma chave só: o nome do nó, e como valor o que ele devolveu. O `next(iter(step.values()))` pega esse valor sem olhar o nome, e assim o rastro aparece na tela enquanto o agente trabalha, em vez de só no fim.

In [ ]:
config = {"configurable": {"thread_id": "assistente"}}

for step in assistant.stream({"messages": [HumanMessage("Quantos arquivos tem nesta pasta?")]}, config, stream_mode="updates"):
    show(next(iter(step.values())))

O modelo escolheu o comando, leu a contagem na observação e escreveu a resposta em português. Nada no grafo diz qual comando rodar nem quantas vezes: uma pergunta que exija dois comandos dá duas voltas no laço.

O laço abaixo lê um pedido por vez com `input` e o manda para a mesma thread, então o assistente lembra das perguntas anteriores. Uma linha vazia encerra. Pergunte, por exemplo, qual desses arquivos é o maior, ou quantas linhas tem um deles.

In [ ]:
def chat(agent, thread_id: str) -> None:
    """Lê pedidos até uma linha vazia e manda cada um para a mesma thread."""
    config = {"configurable": {"thread_id": thread_id}}
    while request := input("você: ").strip():
        for step in agent.stream({"messages": [HumanMessage(request)]}, config, stream_mode="updates"):
            show(next(iter(step.values())))

In [ ]:
chat(assistant, "assistente")

### Exercício 3

Acrescente ao assistente uma ferramenta `write_file`, que grave um arquivo na pasta atual, e converse com ele em uma thread nova: peça que ele crie um arquivo com algum conteúdo e depois que confira, com um comando, que o arquivo existe. Quantas mensagens de ferramenta a thread acumulou no fim?

In [ ]:
# Seu código aqui